### Stage 04 Selection & Pareto Efficiency Analysis

Analyze and compare topic model optimization results from **Stage03** (single or multiple embedding runs).

- **Three selection strategies:**  
  - *Legacy equal weights (50/50)*
  - *Coherence-priority (70/30)*
  - *Current eval_select (40/40/−10/−10)*

Configure runs in `configs/stage04/selection_notebooks.yaml`. Per-run outputs go to `results/selection/{run_id}/notebook_analysis/`; cross-model comparison artifacts go to `comparison.base_dir`.


In [1]:
# --- 0. Setup (run this cell first) ---
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

cwd = Path.cwd().resolve()
project_root = cwd
for _ in range(6):
    if (project_root / "configs").is_dir() and (project_root / "src").is_dir():
        break
    project_root = project_root.parent
else:
    raise RuntimeError("Could not find project root")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.common.config import load_config, resolve_path
from src.legacy.stage04_selection.trials_profile import SUMMARY_SECTIONS, summarize_trials_by_role
from src.stage04_eval_select.notebook_io import (
    LEGACY_RENAME,
    STRATEGY_FILES,
    analyze_pareto_efficiency,
    apply_selection_filters,
    calculate_combined_score,
    compare_metric_snapshot_by_run,
    ensure_run_dirs,
    filter_trials_by_run,
    load_trials_for_runs,
    metric_snapshot,
    normalize_for_pareto,
    normalize_metrics,
    normalize_trials_partial_df,
    resolve_comparison_dirs,
    resolve_run_dirs,
    resolve_runs,
)
from src.stage04_eval_select.weighted_score import apply_weighted_score

METRIC_COLS = ("Coherence", "Topic_Diversity", "n_topics")

nb_cfg = load_config(project_root / "configs" / "selection_notebooks.yaml")
eval_cfg = load_config(project_root / "configs" / "eval_select.yaml")
runs = resolve_runs(nb_cfg)
run_dirs = {run["run_id"]: resolve_run_dirs(run, project_root) for run in runs}
for dirs in run_dirs.values():
    ensure_run_dirs(dirs)

cmp_dirs = resolve_comparison_dirs(nb_cfg, project_root)
multi_run = len(runs) > 1

primary_run = runs[0]
figures_dir = run_dirs[primary_run["run_id"]]["figures_dir"]
tables_dir = run_dirs[primary_run["run_id"]]["tables_dir"]
top_models_dir = run_dirs[primary_run["run_id"]]["top_models_dir"]

sns.set_style(nb_cfg.get("plotting", {}).get("style", "whitegrid"))
plt.rcParams["figure.dpi"] = nb_cfg.get("plotting", {}).get("figure_dpi", 100)
TOP_K = int(nb_cfg["selection"]["top_k"])
PARETO_METRICS = nb_cfg["pareto"]["metrics"]
sel = eval_cfg["selection"]

df_raw = load_trials_for_runs(runs, project_root)
print(f"Project root: {project_root}")
print(f"Runs: {[r['run_id'] for r in runs]} ({'compare' if multi_run else 'single'} mode)")
print(f"Loaded {len(df_raw)} trials × {df_raw.shape[1]} columns")


def cleaning_funnel(steps):
    return pd.DataFrame(steps, columns=["step", "removed", "remaining"])


Project root: /home/polina/Documents/Cursor_Projects/romantic_novels_large_corpus
Runs: ['v3_minilm12v2_first', 'v3_minilm6_first'] (compare mode)
Loaded 260 trials × 28 columns


## 1. Trial overview

Type-aware profile of `df_raw` (loaded in setup). Partial BO rows omit `coherence_c_npmi` / `outlier_rate` (computed later in Stage05 / final `trials.csv`).

In [2]:
for run in runs:
    rid = run["run_id"]
    label = run.get("label", rid)
    subset = df_raw[df_raw["run_id"] == rid]
    print(f"\n{'=' * 60}\nTrial overview — {label} ({rid})\n{'=' * 60}")
    trials_summaries = summarize_trials_by_role(subset)
    for key, title in SUMMARY_SECTIONS:
        if key in trials_summaries:
            print(f"=== {title} ===")
            display(trials_summaries[key])

if multi_run:
    print("\n=== Cross-run summary (raw trials) ===")
    cross = (
        df_raw.groupby(["run_id", "model_label"])
        .agg(
            n_trials=("trial_id", "count"),
            coherence_max=("coherence_c_v", "max"),
            coherence_median=("coherence_c_v", "median"),
            diversity_median=("topic_diversity", "median"),
            stability_pass_pct=("topic_stability_pass", lambda s: float(s.mean()) if len(s) else np.nan),
        )
        .reset_index()
    )
    display(cross)
    if cmp_dirs:
        cross.to_csv(cmp_dirs["tables_dir"] / "compare_run_summary_raw.csv", index=False)

print("\n=== First rows (all runs) ===")
display(df_raw.head())



Trial overview — L12 (v3_minilm12v2_first)


KeyError: "Unclassified trials columns: ['model_label', 'Embeddings_Model']"

## 2. Selection filters

Same row filters as `src.stage04_eval_select.cli select` (`configs/stage04/eval_select.yaml`). All three ranking strategies below share this filtered trial set — no legacy 2σ outlier cleaning.

In [ ]:
filtered_by_run, df_trials, funnel_df = filter_trials_by_run(df_raw, runs, sel)

print("Selection filters (eval_select.yaml) — per run")
for run in runs:
    rid = run["run_id"]
    label = run.get("label", rid)
    sub = funnel_df[funnel_df["run_id"] == rid][["step", "removed", "remaining"]]
    print(f"\n--- {label} ({rid}) ---")
    display(sub)
    renamed = filtered_by_run[rid].rename(columns=LEGACY_RENAME)
    display(metric_snapshot(renamed))

if cmp_dirs:
    funnel_df.to_csv(cmp_dirs["tables_dir"] / "selection_funnel_by_run.csv", index=False)
    compare_snap = compare_metric_snapshot_by_run(df_raw, filtered_by_run, runs)
    compare_snap.to_csv(cmp_dirs["tables_dir"] / "compare_metric_snapshot.csv", index=False)
    print(f"\nSaved comparison tables under {cmp_dirs['tables_dir']}")


## 3. Distribution plots

Before/after filter KDE overlays for coherence and topic diversity. σ reference bands use pre-filter trials (excluding sentinel 0/1 failures).

In [ ]:
from scipy.stats import gaussian_kde

df_plot = df_trials.rename(columns=LEGACY_RENAME)
df_raw_plot = df_raw.rename(columns=LEGACY_RENAME)

STAGE_STYLE = {
    "before": {"color": "steelblue", "kde_ls": "-"},
    "after": {"color": "darkorange", "kde_ls": "--"},
}


def _sigma_reference(before: pd.Series) -> tuple[float, float]:
    """μ and σ from pre-cleaning trials, excluding obvious sentinel failures."""
    ref = before.dropna()
    ref = ref[(ref > 0) & (ref < 1.0)]
    return float(ref.mean()), float(ref.std())


def _shared_bin_edges(before: pd.Series, after: pd.Series, *, max_bins: int = 24) -> np.ndarray:
    """Freedman–Diaconis bins on the combined range, capped for readability."""
    combined = np.concatenate([before.dropna().values, after.dropna().values])
    edges = np.histogram_bin_edges(combined, bins="fd")
    if len(edges) < 10:
        edges = np.histogram_bin_edges(combined, bins=12)
    if len(edges) > max_bins + 1:
        lo, hi = combined.min(), combined.max()
        pad = max((hi - lo) * 0.02, 1e-4)
        edges = np.linspace(lo - pad, hi + pad, max_bins + 1)
    return edges


def _plot_kde_counts(
    ax, values: pd.Series, bin_edges: np.ndarray, *, color: str, linestyle: str
) -> None:
    x = values.dropna().values
    if len(x) < 2:
        return
    bin_width = np.mean(np.diff(bin_edges))
    kde = gaussian_kde(x)
    kde.set_bandwidth(kde.factor * 1.8)  # smoother shape curve for normality read-off
    x_grid = np.linspace(bin_edges[0], bin_edges[-1], 300)
    y = kde(x_grid) * len(x) * bin_width
    ax.plot(x_grid, y, color=color, linewidth=2, linestyle=linestyle, label="_nolegend_")


def plot_before_after_distribution(
    ax,
    before: pd.Series,
    after: pd.Series,
    metric_label: str,
) -> None:
    before = before.dropna()
    after = after.dropna()
    bin_edges = _shared_bin_edges(before, after)
    bin_width = np.mean(np.diff(bin_edges))

    ax.hist(
        before,
        bins=bin_edges,
        alpha=0.45,
        color=STAGE_STYLE["before"]["color"],
        edgecolor="white",
        linewidth=0.6,
        label="Before",
    )
    ax.hist(
        after,
        bins=bin_edges,
        alpha=0.45,
        color=STAGE_STYLE["after"]["color"],
        edgecolor="white",
        linewidth=0.6,
        label="After",
    )

    _plot_kde_counts(
        ax,
        before,
        bin_edges,
        color=STAGE_STYLE["before"]["color"],
        linestyle=STAGE_STYLE["before"]["kde_ls"],
    )
    _plot_kde_counts(
        ax,
        after,
        bin_edges,
        color=STAGE_STYLE["after"]["color"],
        linestyle=STAGE_STYLE["after"]["kde_ls"],
    )

    mu, sigma = _sigma_reference(before)
    sigma_linestyles = {1: ":", 2: "--", 3: "-."}
    for k in (1, 2, 3):
        for sign in (-1, 1):
            ax.axvline(
                mu + sign * k * sigma,
                color="0.45",
                linestyle=sigma_linestyles[k],
                linewidth=1.1,
                alpha=0.9,
                zorder=0,
            )
    ax.axvline(mu, color="black", linestyle="-", linewidth=1.3, alpha=0.7, zorder=0)

    ax.set_title(f"{metric_label} — before (n={len(before)}) vs after (n={len(after)})")
    ax.set_xlabel(metric_label)
    ax.set_ylabel("Frequency")
    ax.set_xlim(bin_edges[0] - bin_width * 0.5, bin_edges[-1] + bin_width * 0.5)

    legend = ax.legend(
        fontsize=7,
        loc="upper left",
        frameon=True,
        framealpha=0.15,
        edgecolor="0.85",
        facecolor="white",
        labelcolor="0.45",
        handlelength=1.4,
        borderpad=0.35,
    )
    for handle in legend.legend_handles:
        handle.set_alpha(0.75)

    print(f"{metric_label} (μ, σ from before cleaning, excl. sentinels 0/1):")
    print(f"  μ={mu:.4f}, σ={sigma:.4f}")
    for k in (1, 2, 3):
        print(f"  ±{k}σ: [{mu - k * sigma:.4f}, {mu + k * sigma:.4f}]")

for run in runs:
    rid = run["run_id"]
    label = run.get("label", rid)
    run_fig = run_dirs[rid]["figures_dir"]
    df_plot = filtered_by_run[rid].rename(columns=LEGACY_RENAME)
    df_raw_plot = df_raw[df_raw["run_id"] == rid].rename(columns=LEGACY_RENAME)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    plot_before_after_distribution(
        axes[0], df_raw_plot["Coherence"], df_plot["Coherence"], f"Coherence ({label})"
    )
    plot_before_after_distribution(
        axes[1], df_raw_plot["Topic_Diversity"], df_plot["Topic_Diversity"], f"Topic Diversity ({label})"
    )
    plt.tight_layout()
    out = run_fig / "distribution_raw_vs_filtered.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    print(f"Saved {out}")
    plt.show()

if multi_run and cmp_dirs:
    last_steps = funnel_df.groupby("run_id").tail(1)
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(data=last_steps, x="model_label", y="remaining", hue="model_label", legend=False, ax=ax)
    ax.set_title("Trials remaining after Stage04 filters")
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.savefig(cmp_dirs["figures_dir"] / "selection_funnel_by_run.png", dpi=150, bbox_inches="tight")
    plt.show()


## 4–5. Strategy A — Equal weights (50/50)

Normalize coherence and topic diversity, then compute a combined score with equal weights (0.5 each). Identify Pareto-efficient trials — points where no other trial is better on both metrics simultaneously — rank the Pareto set by combined score, and export the top **k** models.

In [ ]:
top_equal_by_run = {}
norm_method = nb_cfg["legacy_normalization"].get("method", "zscore")
w_eq = nb_cfg["weighting_strategies"]["equal_weights"]

for run in runs:
    rid = run["run_id"]
    label = run.get("label", rid)
    run_fig = run_dirs[rid]["figures_dir"]
    run_top = run_dirs[rid]["top_models_dir"]
    df_trials_run = filtered_by_run[rid]

    df_legacy = df_trials_run.rename(columns=LEGACY_RENAME)
    df_norm = normalize_metrics(df_legacy.copy(), method=norm_method)
    print(f"\n[{label}] Equal weights — normalization: {norm_method} | trials: {len(df_norm)}")
    print(
        f"  Coherence weight: {w_eq['weight_coherence']:.2f}, "
        f"topic-diversity weight: {w_eq['weight_topic_diversity']:.2f}"
    )

    df_equal = calculate_combined_score(
        df_norm.copy(), float(w_eq["weight_coherence"]), float(w_eq["weight_topic_diversity"])
    )
    df_equal = analyze_pareto_efficiency(
        df_equal, metrics=PARETO_METRICS, per_model=nb_cfg["pareto"].get("analyze_per_model", True)
    )
    pareto_equal = df_equal[df_equal["Pareto_Efficient_All"]]
    top_equal = pareto_equal.nlargest(TOP_K, "Combined_Score")
    top_equal_by_run[rid] = top_equal
    top_equal.to_csv(run_top / STRATEGY_FILES["equal_weights"], index=False)
    print(f"  Pareto-efficient: {len(pareto_equal)} | Saved top {len(top_equal)}")

    display(
        top_equal[["trial_id", "Embeddings_Model", "Coherence", "Topic_Diversity", "Combined_Score"]].head(TOP_K)
    )

    fig, ax = plt.subplots(figsize=(12, 8))
    sns.scatterplot(
        data=df_equal, x="Topic_Diversity", y="Coherence", hue="Embeddings_Model",
        palette="Set2", s=70, alpha=0.7, ax=ax,
    )
    ax.scatter(
        pareto_equal["Topic_Diversity"], pareto_equal["Coherence"],
        facecolors="none", edgecolors="red", s=200, linewidths=2, label="Pareto-efficient", zorder=10,
    )
    ax.set_title(f"Pareto Front — Equal Weights ({label})", fontsize=14)
    ax.set_xlabel("Topic Diversity")
    ax.set_ylabel("Coherence")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(run_fig / "pareto_front_equal_weights.png", dpi=150, bbox_inches="tight")
    plt.show()

top_equal = top_equal_by_run[primary_run["run_id"]]


## 6. Strategy B — Coherence priority (70/30)

Reuses z-scored metrics from Strategy A, but ranks with **70% coherence / 30% topic diversity**. Pareto efficiency is computed on the same normalized axes (weights do not change the frontier); only the ranking within the Pareto set differs from Strategy A.

In [ ]:
top_priority_by_run = {}
w_cp = nb_cfg["weighting_strategies"]["coherence_priority"]

for run in runs:
    rid = run["run_id"]
    label = run.get("label", rid)
    run_fig = run_dirs[rid]["figures_dir"]
    run_top = run_dirs[rid]["top_models_dir"]
    df_trials_run = filtered_by_run[rid]

    df_legacy = df_trials_run.rename(columns=LEGACY_RENAME)
    df_norm = normalize_metrics(df_legacy.copy(), method=norm_method)
    df_priority = calculate_combined_score(
        df_norm.copy(), float(w_cp["weight_coherence"]), float(w_cp["weight_topic_diversity"])
    )
    df_priority = analyze_pareto_efficiency(df_priority, metrics=PARETO_METRICS, per_model=False)
    pareto_priority = df_priority[df_priority["Pareto_Efficient_All"]]
    top_priority = pareto_priority.nlargest(TOP_K, "Combined_Score")
    top_priority_by_run[rid] = top_priority
    top_priority.to_csv(run_top / STRATEGY_FILES["coherence_priority"], index=False)
    print(f"\n[{label}] Coherence priority — Pareto: {len(pareto_priority)} | Saved top {len(top_priority)}")

    display(
        top_priority[["trial_id", "Embeddings_Model", "Coherence", "Topic_Diversity", "Combined_Score"]].head(TOP_K)
    )

    fig, ax = plt.subplots(figsize=(12, 8))
    sns.scatterplot(
        data=df_priority, x="Topic_Diversity", y="Coherence", hue="Embeddings_Model",
        palette="Set2", s=70, alpha=0.7, ax=ax,
    )
    ax.scatter(
        pareto_priority["Topic_Diversity"], pareto_priority["Coherence"],
        facecolors="none", edgecolors="red", s=200, linewidths=2, label="Pareto-efficient", zorder=10,
    )
    ax.set_title(f"Pareto Front — Coherence Priority ({label})", fontsize=14)
    ax.set_xlabel("Topic Diversity")
    ax.set_ylabel("Coherence")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(run_fig / "pareto_front_coherence_priority.png", dpi=150, bbox_inches="tight")
    plt.show()

top_priority = top_priority_by_run[primary_run["run_id"]]


## 7. Strategy C — eval_select (current pipeline)

Mirrors `src.stage04_eval_select.cli select`: **min-max** normalize coherence and topic diversity to [0, 1], keep Pareto-efficient trials, then rank survivors with the production weighted objective (40/40 coherence & diversity, minus 10% outlier rate and 10% stability penalty). Validates notebook output against CLI `top_k.csv` when present.

In [ ]:
top_eval_by_run = {}
weights = eval_cfg["weights"]
print(
    "Weighted-score weights (eval_select.yaml): "
    f"coherence={weights['coherence']}, diversity={weights['diversity']}, "
    f"outlier={weights['outlier']}, stability={weights['stability']}"
)

for run in runs:
    rid = run["run_id"]
    label = run.get("label", rid)
    run_fig = run_dirs[rid]["figures_dir"]
    run_top = run_dirs[rid]["top_models_dir"]
    cli_top_k_path = resolve_path(Path(run["inputs"]["eval_select_top_k"]), project_root)
    df_trials_run = filtered_by_run[rid]

    df_eval = normalize_for_pareto(df_trials_run.copy())
    df_eval_p = df_eval.rename(
        columns={
            "coherence_c_v_norm": "Coherence_norm",
            "topic_diversity_norm": "Topic_Diversity_norm",
            "embedding_model": "Embeddings_Model",
        }
    )
    df_eval_p = analyze_pareto_efficiency(df_eval_p, metrics=PARETO_METRICS, per_model=False)
    candidates = df_eval_p[df_eval_p["Pareto_Efficient_All"]].copy()
    candidates = apply_weighted_score(
        candidates,
        float(weights["coherence"]),
        float(weights["diversity"]),
        float(weights["outlier"]),
        float(weights["stability"]),
    )
    candidates = candidates.sort_values("weighted_score", ascending=False)
    top_eval = candidates.head(TOP_K)
    top_eval_by_run[rid] = top_eval
    top_eval.to_csv(run_top / STRATEGY_FILES["eval_select"], index=False)
    print(f"\n[{label}] eval_select — Pareto candidates: {len(candidates)} | Saved top {len(top_eval)}")

    display(
        top_eval[["trial_id", "Embeddings_Model", "coherence_c_v", "topic_diversity", "weighted_score"]].head(TOP_K)
    )

    if cli_top_k_path.exists():
        cli_top = pd.read_csv(cli_top_k_path)
        winner_match = top_eval.iloc[0]["trial_id"] == cli_top.iloc[0]["trial_id"]
        overlap = len(set(top_eval["trial_id"]) & set(cli_top["trial_id"]))
        score_ok = abs(float(top_eval.iloc[0]["weighted_score"]) - float(cli_top.iloc[0]["weighted_score"])) < 1e-6
        print(f"  CLI validation — winner match: {winner_match}, overlap: {overlap}/{len(cli_top)}, score match: {score_ok}")
    else:
        print(f"  CLI top_k not found: {cli_top_k_path}")

    fig, ax = plt.subplots(figsize=(12, 8))
    sns.scatterplot(
        data=df_eval, x="topic_diversity", y="coherence_c_v", hue="embedding_model",
        palette="Set2", s=70, alpha=0.7, ax=ax,
    )
    pareto_v3 = df_eval_p[df_eval_p["Pareto_Efficient_All"]]
    ax.scatter(
        pareto_v3["topic_diversity"], pareto_v3["coherence_c_v"],
        facecolors="none", edgecolors="red", s=200, linewidths=2, label="Pareto-efficient", zorder=10,
    )
    ax.set_title(f"Pareto Front — eval_select ({label})", fontsize=14)
    ax.set_xlabel("Topic Diversity")
    ax.set_ylabel("Coherence (c_v)")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(run_fig / "pareto_front_eval_select.png", dpi=150, bbox_inches="tight")
    plt.show()

top_eval = top_eval_by_run[primary_run["run_id"]]


## 7b. Cross-model comparison

Overlay filtered trials and top-1 winners per run (when `runs` has more than one entry). Outputs go to `comparison.base_dir` in `configs/stage04/selection_notebooks.yaml`.

In [ ]:
if multi_run and cmp_dirs:
    # Overlay all filtered trials
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.scatterplot(
        data=df_trials,
        x="topic_diversity",
        y="coherence_c_v",
        hue="model_label",
        palette="Set1",
        s=70,
        alpha=0.75,
        ax=ax,
    )
    winners = []
    for run in runs:
        rid = run["run_id"]
        w = top_eval_by_run[rid].iloc[0]
        winners.append(w)
        ax.scatter(
            w["topic_diversity"], w["coherence_c_v"],
            s=220, marker="*", edgecolors="black", linewidths=1.2, zorder=12,
            label=f"{run.get('label', rid)} winner",
        )
    ax.set_title("Filtered trials + eval_select winners (all runs)")
    ax.set_xlabel("Topic Diversity")
    ax.set_ylabel("Coherence (c_v)")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(cmp_dirs["figures_dir"] / "pareto_overlay_eval_select.png", dpi=150, bbox_inches="tight")
    plt.show()

    winner_rows = []
    for run in runs:
        rid = run["run_id"]
        for strategy, top_map in [
            ("equal_weights", top_equal_by_run),
            ("coherence_priority", top_priority_by_run),
            ("eval_select", top_eval_by_run),
        ]:
            top_df = top_map[rid]
            row = top_df.iloc[0].to_dict()
            row["strategy"] = strategy
            row["run_id"] = rid
            row["model_label"] = run.get("label", rid)
            winner_rows.append(row)
    winners_df = pd.DataFrame(winner_rows)
    winners_path = cmp_dirs["tables_dir"] / "winners_by_run_and_strategy.csv"
    winners_df.to_csv(winners_path, index=False)
    print(f"Saved {winners_path}")
    display(
        winners_df[
            [c for c in ["model_label", "strategy", "trial_id", "coherence_c_v", "topic_diversity", "n_topics", "weighted_score", "Combined_Score"] if c in winners_df.columns]
        ]
    )
else:
    print("Single-run mode: skipping cross-model comparison section.")


## 8. Cross-strategy comparison

Compare top-**k** trial sets from all three strategies: pairwise overlap counts, union size, **per-trial parameter dumps** (metrics, `n_topics`, hyperparameters), and a scatter plot tagging each trial by which strategy(ies) selected it (A = equal weights, B = coherence priority, C = eval_select).

In [ ]:
for run in runs:
    rid = run["run_id"]
    label = run.get("label", rid)
    run_tbl = run_dirs[rid]["tables_dir"]
    run_top = run_dirs[rid]["top_models_dir"]
    df_trials_run = filtered_by_run[rid]

    top_equal_run = top_equal_by_run[rid]
    top_priority_run = top_priority_by_run[rid]
    top_eval_run = top_eval_by_run[rid]

    sets = {
        "equal_weights": set(top_equal_run["trial_id"]),
        "coherence_priority": set(top_priority_run["trial_id"]),
        "eval_select": set(top_eval_run["trial_id"]),
    }

    rows = []
    for a in sets:
        for b in sets:
            if a != b:
                rows.append({"strategy": a, "compare_to": b, "overlap_count": len(sets[a] & sets[b])})
    overlap_df = pd.DataFrame(rows)
    overlap_df.to_csv(run_tbl / "strategy_pairwise_overlap.csv", index=False)

    summary_df = pd.DataFrame(
        [{"strategy": k, "top_k_count": len(v), "trial_ids": ";".join(sorted(v))} for k, v in sets.items()]
    )
    summary_df.to_csv(run_top / "strategy_overlap_summary.csv", index=False)

    print(f"\n{'=' * 60}\nCross-strategy comparison — {label} ({rid})\n{'=' * 60}")
    print("Top-k sizes:", {k: len(v) for k, v in sets.items()})
    print(f"Union of all selected trials: {len(set().union(*sets.values()))}")
    display(overlap_df)
    display(summary_df)

    _param_cols = (
        ["trial_id", "embedding_model", "coherence_c_v", "topic_diversity", "n_topics"]
        + list(nb_cfg["hyperparameters"])
        + [
            "n_topics_std", "n_topics_min", "n_topics_max", "n_topics_runs",
            "stability_score", "topic_stability_pass", "bo_objective", "seed", "bo_call",
        ]
    )
    param_cols = [c for c in _param_cols if c in df_trials_run.columns]

    _strategy_tops = [
        ("equal_weights (A)", top_equal_run, "Combined_Score"),
        ("coherence_priority (B)", top_priority_run, "Combined_Score"),
        ("eval_select (C)", top_eval_run, "weighted_score"),
    ]

    detail_parts = []
    for strategy_label, top_df, score_col in _strategy_tops:
        print(f"\n{strategy_label} — {len(top_df)} selected trial(s)")
        ordered_ids = [tid for tid in top_df["trial_id"].tolist() if tid in set(df_trials_run["trial_id"])]
        block = df_trials_run[df_trials_run["trial_id"].isin(ordered_ids)].copy()
        block["order_tmp"] = block["trial_id"].apply(lambda x: ordered_ids.index(x))
        block = block.sort_values("order_tmp").drop(columns="order_tmp")
        block = block[param_cols]
        block["strategy"] = strategy_label
        block["rank"] = range(1, len(block) + 1)
        if score_col in top_df.columns:
            score_map = dict(zip(top_df["trial_id"], top_df[score_col]))
            block["strategy_score"] = [score_map.get(tid) for tid in ordered_ids]
        detail_parts.append(block)

    selected_details = pd.concat(detail_parts, ignore_index=True)
    details_path = run_tbl / "strategy_selected_trial_details.csv"
    selected_details.to_csv(details_path, index=False)
    print(f"Saved trial details: {details_path}")

    plot_df = df_raw[df_raw["run_id"] == rid].copy()
    label_map = {}
    for tid in plot_df["trial_id"]:
        tags = [
            t for t, s in [("A", "equal_weights"), ("B", "coherence_priority"), ("C", "eval_select")]
            if tid in sets[s]
        ]
        label_map[tid] = "+".join(tags) if tags else "none"
    plot_df["selected_by"] = plot_df["trial_id"].map(label_map)

    fig, ax = plt.subplots(figsize=(12, 8))
    for memb, group in plot_df.groupby("selected_by"):
        ax.scatter(group["topic_diversity"], group["coherence_c_v"], label=memb, alpha=0.7, s=50)
    ax.set_title(f"Strategy Membership — {label}", fontsize=14)
    ax.set_xlabel("Topic Diversity")
    ax.set_ylabel("Coherence (c_v)")
    ax.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left", title="Selected by")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(run_dirs[rid]["figures_dir"] / "strategy_membership_scatter.png", dpi=150, bbox_inches="tight")
    plt.show()


## 9. Hyperparameter preview

Descriptive statistics (mean, std, min, max, quartiles) for configured BERTopic/HDBSCAN/UMAP/vectorizer hyperparameters within each strategy's top-**k** export.

In [ ]:
hp_cols = nb_cfg["hyperparameters"]
preview_parts = []

for run in runs:
    rid = run["run_id"]
    label = run.get("label", rid)
    for name, df in [
        ("equal_weights", top_equal_by_run[rid]),
        ("coherence_priority", top_priority_by_run[rid]),
        ("eval_select", top_eval_by_run[rid]),
    ]:
        available = [c for c in hp_cols if c in df.columns]
        s = df[available].describe().T
        s["strategy"] = name
        s["run_id"] = rid
        s["model_label"] = label
        preview_parts.append(s.reset_index().rename(columns={"index": "hyperparameter"}))

preview = pd.concat(preview_parts, ignore_index=True)
for run in runs:
    rid = run["run_id"]
    sub = preview[preview["run_id"] == rid]
    sub.to_csv(run_dirs[rid]["tables_dir"] / "hyperparameter_descriptive_preview.csv", index=False)

if cmp_dirs and multi_run:
    preview.to_csv(cmp_dirs["tables_dir"] / "hyperparameter_descriptive_preview_all_runs.csv", index=False)

print(f"Hyperparameters configured: {len(hp_cols)} | preview rows: {len(preview)}")
display(preview.head(20))
